#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [51]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_5mm.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [52]:
def guardar_cajas_y_productos(grosor):
    caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")
    cajas = [
        Caja(
            caja_id = row["caja_tipo_id"],
            dim_interior_ancho = row["caja_interior_ancho"],
            dim_interior_largo = row["caja_interior_largo"],
            dim_interior_alto = row["caja_interior_alto"],
            costo_unitario = row['costo_unitario_base']
        )
        for _, row in caja_compras_merge.iterrows()
    ]

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    # Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [53]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

Armamos una función general para buscar un tipo de caja por su id.

In [54]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

In [55]:
solucion_base = Solucion(grosor=5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto,caja)
        solucion_base.agregar_asignacion(asignacion, descuentos=False)

solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,02cf77de65b70dd77905e2e33d78478f,0.323810,0.983137,0.0,0.0,0.0,0.0,0.0,927967.80,103109,15466350,16394317.80
1,BR0001,1546613,72,0835ff365412a67b720a19713ec250f3,0.937728,1.089928,0.0,0.0,0.0,0.0,0.0,1005298.45,32223,4833450,5838748.45
2,BR0001,1546613,72,125888817c4da192ce9335454b8d4432,0.883581,1.158994,0.0,0.0,0.0,0.0,0.0,1082629.10,32223,4833450,5916079.10
3,BR0001,1546613,72,170f5916eaeb0e271ae012eedb1ad645,0.835707,1.016779,0.0,0.0,0.0,0.0,0.0,1005298.45,38667,5800050,6805348.45
4,BR0001,1546613,72,1fe6b9010b30d1d429fc0c7a240159aa,0.342098,1.121933,0.0,0.0,0.0,0.0,0.0,927967.80,85924,12888600,13816567.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27129,BR0421,145803,56,f152ca1c81c447e9f47aac4e934dc6fc,0.912094,1.285170,0.0,0.0,0.0,0.0,0.0,102062.10,1823,273450,375512.10
27130,BR0421,145803,56,f95687311339e4b9b0ef5622b26fd941,0.433113,1.071770,0.0,0.0,0.0,0.0,0.0,87481.80,4557,683550,771031.80
27131,BR0421,145803,56,fa8da38322149b23e14a120e71d38975,0.423966,0.953191,0.0,0.0,0.0,0.0,0.0,87481.80,5208,781200,868681.80
27132,BR0421,145803,56,fc6931875b8384c7bacd18bb4699795f,0.396279,1.022831,0.0,0.0,0.0,0.0,0.0,87481.80,5208,781200,868681.80


#### **Greedy 1: Elección por menor costo**

In [56]:
def solucion_greedy(grosor, cajas, productos_ordenados, criterio):

    solucion = Solucion(grosor=grosor)

    for producto in productos_ordenados:
        metricas = {}  # caja_id -> (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
        
        for caja_id in producto.cajas_asignables:
            caja = buscar_caja_por_id(caja_id)
            
            # Volumen requerido actual (antes de asignar este producto)
            volumen_total = caja.unidades_total_requeridas()
            utilizacion_pallet = caja.utilizacion_pallet()
            
            # Costo simulado de asignar este producto a esta caja
            asignacion = Asignacion(producto, caja)
            caja.asignar_producto(producto)
            
            # Calculamos los costos del producto utilizando ese tipo de caja
            costo_packaging = asignacion.costo_packaging_producto_total()
            costo_flete = 150 * asignacion.cant_pallets_requeridas()
            costo_total = costo_packaging + costo_flete
            metricas[caja_id] = (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
            
            # Revertimos la asignación para que no se aplique un descuento posterior
            caja.revocar_producto(producto)
        
        if criterio == "minimizar_costo_total":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][3]))
        elif criterio == "maximizar_volumen_tipo":        
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][3]))
        elif criterio == "minimizar_costo_flete":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][2]))
        elif criterio == "maximizar_utilizacion_pallet":
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][4]))
        
        caja_optima = buscar_caja_por_id(caja_id_optima)
        
        asignacion = Asignacion(producto, caja_optima)
        solucion.agregar_asignacion(asignacion)    
    
    return solucion

Greedy (1)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con menor costo considerando el cálculo de descuentos sumando los volúmenes de producto
    Recalculo descuentos

Con los descuentos finales, vuelvo a calcular cada costo de producto

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [57]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy1 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()
solucion_greedy1.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 23
Costo packaging: 29867179.525000006
Costo flete: 131926350
Costo total: 161793529.525
Utilización de pallet promedio: 0.918306000607566
Utilización de caja promedio: 1.2634173649495755


In [58]:
solucion_greedy1.resumen_por_asignacion()


,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,5914abeae491eab8612131985b15994f,0.945868,1.267782,-0.3,-0.3,-0.3,-0.1,-0.2,715185.835,27619,4142850,4858035.835
1,BR0002,139211,83,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.271230,-0.3,-0.3,-0.3,0.0,-0.3,63341.005,2176,326400,389741.005
2,BR0003,172506,43,3068e6f2a2ce79314e458b7a6ab57b4d,0.930163,1.138889,-0.3,-0.3,-0.2,-0.2,-0.3,78490.230,1961,294150,372640.230
3,BR0004,271715,88,228ba4e5ea3c7574cbd9ce77434d4525,0.925275,1.321090,-0.2,-0.3,-0.2,-0.3,-0.3,133140.350,3774,566100,699240.350
4,BR0005,7586,106,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.167981,-0.3,-0.3,-0.3,0.0,-0.3,3451.630,119,17850,21301.630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1256.255,39,5850,7106.255
423,BR0418,3942,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1793.610,55,8250,10043.610
424,BR0419,43300,54,3068e6f2a2ce79314e458b7a6ab57b4d,0.930163,1.257037,-0.3,-0.3,-0.2,-0.2,-0.3,22516.000,493,73950,96466.000
425,BR0420,205852,41,519a80871f454928779db9ed3e039e2b,0.933183,1.142388,-0.2,-0.3,-0.3,-0.3,-0.3,100867.480,2574,386100,486967.480


In [59]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy2 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy2.resumen_por_asignacion()
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 27
Costo packaging: 30305007.99000003
Costo flete: 131892150
Costo total: 162197157.99000004
Utilización de pallet promedio: 0.9234925447347336
Utilización de caja promedio: 1.2558748412249592


In [60]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy3 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy3.resumen_por_asignacion()
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 17
Costo packaging: 29760015.04
Costo flete: 155073300
Costo total: 184833315.04
Utilización de pallet promedio: 0.9529277809508822
Utilización de caja promedio: 1.0302309985046123


In [61]:
solucion_greedy3.resumen_por_asignacion()


,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,2060590a59a5c59ff1bb32dbffe1028b,0.970288,1.052083,0.1,-0.3,-0.3,-0.1,-0.2,715185.835,32223,4833450,5548635.835
1,BR0002,139211,83,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.271230,-0.3,-0.3,-0.3,-0.3,-0.3,63341.005,2176,326400,389741.005
2,BR0003,172506,43,67894c5a8355ca9b7e57b96362fa45f1,0.982876,0.970414,-0.2,-0.3,-0.2,0.0,-0.2,89703.120,2157,323550,413253.120
3,BR0004,271715,88,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.095214,-0.3,-0.3,-0.3,-0.3,-0.3,123630.325,4246,636900,760530.325
4,BR0005,7586,106,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.167981,-0.3,-0.3,-0.3,-0.3,-0.3,3451.630,119,17850,21301.630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,ea6403e25e3a52198dad19251c207a47,0.844404,0.947362,-0.3,-0.3,-0.3,-0.2,0.1,1352.890,50,7500,8852.890
423,BR0418,3942,50,ea6403e25e3a52198dad19251c207a47,0.844404,0.947362,-0.3,-0.3,-0.3,-0.2,0.1,1931.580,71,10650,12581.580
424,BR0419,43300,54,161332db17a4c8bcba8a5651098cbe91,0.983425,0.957743,-0.3,-0.3,-0.2,-0.2,-0.3,22516.000,602,90300,112816.000
425,BR0420,205852,41,1e68edd4dd3edf793634e204ec225c3b,0.949000,0.887343,-0.2,-0.3,-0.3,-0.3,-0.3,100867.480,3217,482550,583417.480


In [62]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy4 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy4.resumen_por_asignacion()
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 8
Costo packaging: 29469724.869999994
Costo flete: 150772200
Costo total: 180241924.87
Utilización de pallet promedio: 0.8653385973325733
Utilización de caja promedio: 1.1722419084808986


In [63]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy5 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy5.resumen_por_asignacion()
solucion_greedy5.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 24
Costo packaging: 29960784.419999987
Costo flete: 131895150
Costo total: 161855934.42
Utilización de pallet promedio: 0.9200925438866193
Utilización de caja promedio: 1.261209310487083


In [64]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy6 = solucion_greedy(5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy6.resumen_por_asignacion()
solucion_greedy6.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 27
Costo packaging: 30305007.99000001
Costo flete: 131892150
Costo total: 162197157.99
Utilización de pallet promedio: 0.9234925447347335
Utilización de caja promedio: 1.2558748412249596


In [65]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy7 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy7.resumen_por_asignacion()
solucion_greedy7.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 17
Costo packaging: 29760015.039999984
Costo flete: 155073300
Costo total: 184833315.04
Utilización de pallet promedio: 0.9529277809508823
Utilización de caja promedio: 1.0302309985046119


In [66]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy8 = solucion_greedy(5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy8.resumen_por_asignacion()
solucion_greedy8.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 10
Costo packaging: 29533654.199999977
Costo flete: 144668850
Costo total: 174202504.2
Utilización de pallet promedio: 0.9023581694264049
Utilización de caja promedio: 1.1736096055855718


#### **Greedy 2: Elección por mayor volumen**

Greedy (2)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con mayor volúmen actualmente

Con los descuentos finales, vuelvo a calcular cada costo de producto

In [67]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

In [68]:
solucion_greedy2 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy2.agregar_asignacion(asignacion)

solucion_greedy2.exportar_submmit(nombre_csv="3-greedy2_5mm")
solucion_greedy2.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,2060590a59a5c59ff1bb32dbffe1028b,0.970288,1.052083,-0.3,-0.3,-0.3,-0.1,-0.3,715185.835,32223,4833450,5548635.835
1,BR0002,139211,83,63849e2a342c002e41cdca735b7ffa35,0.887333,1.231162,-0.3,-0.3,-0.3,-0.3,-0.3,68213.390,2486,372900,441113.390
2,BR0003,172506,43,8f6cdeebd6a09f639e27ef1cbb567311,0.769843,1.252844,-0.3,-0.3,-0.3,-0.2,-0.3,78490.230,2157,323550,402040.230
3,BR0004,271715,88,63849e2a342c002e41cdca735b7ffa35,0.887333,1.060694,-0.3,-0.3,-0.3,-0.3,-0.3,133140.350,4853,727950,861090.350
4,BR0005,7586,106,63849e2a342c002e41cdca735b7ffa35,0.887333,1.131168,-0.3,-0.3,-0.3,-0.3,-0.3,3717.140,136,20400,24117.140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.3,-0.3,1256.255,39,5850,7106.255
423,BR0418,3942,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.3,-0.3,1793.610,55,8250,10043.610
424,BR0419,43300,54,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.126737,-0.3,-0.3,-0.3,-0.3,-0.3,19701.500,602,90300,110001.500
425,BR0420,205852,41,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.138808,-0.3,-0.3,-0.3,-0.3,-0.3,93662.660,2860,429000,522662.660


In [69]:
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 8
Costo packaging: 29469724.869999994
Costo flete: 150772200
Costo total: 180241924.87
Utilización de pallet promedio: 0.8653385973325733
Utilización de caja promedio: 1.1722419084808986


#### **Greedy 3: Ordenamiento de productos según su demanda total**

Greedy (3)

Rehacer Greedy 1 y 2 ordenando los productos de mayor a menor según la demanda.

In [70]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

# Ordenamos ahora los productos según la demanda total de mayor a menor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

In [71]:
solucion_greedy3 = Solucion(grosor=5)

for producto in productos_ordenados:
    costos_totales = {}
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        
        costo_total = costo_packaging + costo_flete
        costos_totales[caja_id] = costo_total
        
        caja.revocar_producto(producto)
    
    caja_id_optima = min(costos_totales, key=costos_totales.get)
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy3.agregar_asignacion(asignacion)
    
solucion_greedy3.exportar_submmit(nombre_csv="4-greedy3_5mm")
solucion_greedy3.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,414d4cf5c1bc8a346f7e4445ede7acbd,0.907881,1.323144,0.1,-0.3,-0.2,-0.1,0.1,716913.730,27619,4142850,4859763.730
1,BR0002,139211,83,5229dfb764d843b66341f7ca5cdec3dd,0.983974,1.271230,-0.3,-0.3,-0.3,-0.1,-0.2,63341.005,2176,326400,389741.005
2,BR0003,172506,43,eb8c91d6266d086a541d9c7411a9d92f,0.960363,1.100671,-0.2,-0.2,-0.3,0.1,-0.3,78490.230,1961,294150,372640.230
3,BR0004,271715,88,228ba4e5ea3c7574cbd9ce77434d4525,0.925275,1.321090,-0.2,-0.3,-0.2,-0.3,-0.3,133140.350,3774,566100,699240.350
4,BR0005,7586,106,161332db17a4c8bcba8a5651098cbe91,0.983425,1.322476,-0.1,-0.2,-0.3,0.1,-0.2,3944.720,106,15900,19844.720
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1256.255,39,5850,7106.255
423,BR0418,3942,50,082c1cdb42b1abd201403ca33ca11ef0,0.840725,1.238095,-0.3,-0.3,-0.3,-0.2,-0.2,1793.610,55,8250,10043.610
424,BR0419,43300,54,3068e6f2a2ce79314e458b7a6ab57b4d,0.930163,1.257037,-0.3,-0.3,-0.2,-0.2,0.0,22516.000,493,73950,96466.000
425,BR0420,205852,41,519a80871f454928779db9ed3e039e2b,0.933183,1.142388,-0.2,-0.3,-0.3,-0.3,-0.3,100867.480,2574,386100,486967.480


In [72]:
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 24
Costo packaging: 29960784.419999987
Costo flete: 131895150
Costo total: 161855934.42
Utilización de pallet promedio: 0.9200925438866193
Utilización de caja promedio: 1.261209310487083


In [73]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

In [74]:
solucion_greedy4 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy4.agregar_asignacion(asignacion)

solucion_greedy4.exportar_submmit(nombre_csv="5-greedy4_5mm")
solucion_greedy4.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,72,5914abeae491eab8612131985b15994f,0.945868,1.267782,-0.3,-0.3,-0.3,-0.3,-0.3,703708.915,27619,4142850,4846558.915
1,BR0002,139211,83,5914abeae491eab8612131985b15994f,0.945868,1.152155,-0.3,-0.3,-0.3,-0.3,-0.3,63341.005,2486,372900,436241.005
2,BR0003,172506,43,519a80871f454928779db9ed3e039e2b,0.933183,1.024045,-0.3,-0.3,-0.3,-0.3,-0.3,84527.940,2157,323550,408077.940
3,BR0004,271715,88,5914abeae491eab8612131985b15994f,0.945868,0.992626,-0.3,-0.3,-0.3,-0.3,-0.3,123630.325,4853,727950,851580.325
4,BR0005,7586,106,5914abeae491eab8612131985b15994f,0.945868,1.058577,-0.3,-0.3,-0.3,-0.3,-0.3,3451.630,136,20400,23851.630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,50,1a3ffa8018de4637afcdd1e390e2602a,0.784865,1.175879,-0.3,-0.3,-0.3,-0.3,-0.3,1256.255,44,6600,7856.255
423,BR0418,3942,50,1a3ffa8018de4637afcdd1e390e2602a,0.784865,1.175879,-0.3,-0.3,-0.3,-0.3,-0.3,1793.610,62,9300,11093.610
424,BR0419,43300,54,519a80871f454928779db9ed3e039e2b,0.933183,1.130279,-0.3,-0.3,-0.3,-0.3,-0.3,21217.000,542,81300,102517.000
425,BR0420,205852,41,519a80871f454928779db9ed3e039e2b,0.933183,1.142388,-0.3,-0.3,-0.3,-0.3,-0.3,100867.480,2574,386100,486967.480


In [75]:
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 5mm
Número de tipos de cajas distintos: 10
Costo packaging: 29533654.199999977
Costo flete: 144668850
Costo total: 174202504.2
Utilización de pallet promedio: 0.9023581694264049
Utilización de caja promedio: 1.1736096055855718


Partiendo de la solucion, veo si la puedo mejorar

En cada paso:
    Cambio el tipo de caja, y me fijo si con los descuentos puedo tener un menor costo